# 2. Entrenamiento del modelo — Deteccion de fraude

Problema: **clasificacion binaria supervisada** (`isFraud`), fuertemente
desbalanceado (~0.1-0.3% de positivos).

Flujo:
1. Train/test split estratificado sobre la **referencia** (`step <= 400`).
2. Modelo base (Logistic Regression) como piso de comparacion.
3. Optimizacion de hiperparametros de un `XGBoostClassifier`:
   - **Random Search** (baseline de busqueda).
   - **Optimizacion Bayesiana con Optuna** (tecnica avanzada, Sesion 4).
4. Registro de experimentos (parametros, metricas, artefactos) en **MLflow**.
5. Evaluacion **offline** (holdout de referencia) y **online** (datos de
   produccion `step > 400`, que en este dataset simulado si tienen
   `isFraud` real, permitiendo medir el desempeno "como si" ya estuviera
   desplegado).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

from src.config import (
    MODEL_DIR,
    MODEL_PATH,
    PRODUCTION_DATA_PATH,
    RANDOM_STATE,
    REFERENCE_DATA_PATH,
    TARGET_COL,
)
from src.features import select_model_features
from src.metrics import classification_metrics
from src.tracking import init_mlflow, log_classification_metrics

MODEL_DIR.mkdir(parents=True, exist_ok=True)
init_mlflow()

In [ ]:
reference = pd.read_csv(REFERENCE_DATA_PATH)
production = pd.read_csv(PRODUCTION_DATA_PATH)

FEATURES = select_model_features(reference)
print(f"{len(FEATURES)} features:", FEATURES)

X = reference[FEATURES]
y = reference[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## 2.1 Modelo base (Logistic Regression)

Sirve como piso de comparacion antes de optimizar un modelo mas complejo.

In [ ]:
with mlflow.start_run(run_name="baseline_logreg"):
    baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    baseline.fit(X_train, y_train)

    proba = baseline.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    metrics = classification_metrics(y_test, preds, proba)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    log_classification_metrics("offline_", metrics)
    print(metrics)

## 2.2 Optimizacion de hiperparametros — XGBoost

Comparamos dos tecnicas vistas en el curso (Sesion 3 y 4):

- **Random Search** (`RandomizedSearchCV`): exploracion aleatoria del
  espacio de hiperparametros.
- **Optimizacion Bayesiana** (`optuna`): construye un modelo probabilistico
  del espacio de busqueda y propone el siguiente punto a evaluar en funcion
  de los resultados anteriores -> converge mas rapido a buenas regiones.

Se usa `scale_pos_weight` para compensar el fuerte desbalance de clases en
vez de sobre/sub-muestrear, y PR-AUC como metrica objetivo (mas informativa
que accuracy/ROC-AUC bajo clases raras).

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_dist = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_search = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_dist,
    n_iter=15,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
random_search.fit(X_train, y_train)
print("Mejor PR-AUC (CV) Random Search:", random_search.best_score_)
print(random_search.best_params_)

In [ ]:
def optuna_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )
    scores = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        val_proba = model.predict_proba(X_train.iloc[val_idx])[:, 1]
        scores.append(classification_metrics(
            y_train.iloc[val_idx], (val_proba >= 0.5).astype(int), val_proba
        )["pr_auc"])
    return float(np.mean(scores))


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(optuna_objective, n_trials=25, show_progress_bar=True)
print("Mejor PR-AUC (CV) Optuna:", study.best_value)
print(study.best_params)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
optuna.visualization.matplotlib.plot_optimization_history(study, ax=ax[0])
optuna.visualization.matplotlib.plot_param_importances(study, ax=ax[1])
plt.tight_layout()
plt.show()

## 2.3 Modelo campeon ('champion')

Se elige el mejor resultado entre Random Search y Optuna, y se entrena sobre todo el train set.

In [ ]:
candidates = {
    "random_search": (random_search.best_score_, random_search.best_params_),
    "optuna": (study.best_value, study.best_params),
}
best_method = max(candidates, key=lambda k: candidates[k][0])
best_params = candidates[best_method][1]
print("Metodo ganador:", best_method, "->", best_params)

champion = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params,
)
champion.fit(X_train, y_train)

In [ ]:
with mlflow.start_run(run_name=f"champion_xgboost_{best_method}"):
    mlflow.log_param("model", "XGBClassifier")
    mlflow.log_param("hpo_method", best_method)
    for k, v in best_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    # --- Evaluacion OFFLINE (holdout de referencia, step <= 400) ---
    proba_test = champion.predict_proba(X_test)[:, 1]
    preds_test = (proba_test >= 0.5).astype(int)
    offline_metrics = classification_metrics(y_test, preds_test, proba_test)
    log_classification_metrics("offline_", offline_metrics)
    print("OFFLINE:", offline_metrics)

    # --- Evaluacion ONLINE (produccion, step > 400) ---
    X_prod, y_prod = production[FEATURES], production[TARGET_COL]
    proba_prod = champion.predict_proba(X_prod)[:, 1]
    preds_prod = (proba_prod >= 0.5).astype(int)
    online_metrics = classification_metrics(y_prod, preds_prod, proba_prod)
    log_classification_metrics("online_", online_metrics)
    print("ONLINE:", online_metrics)

    mlflow.xgboost.log_model(champion, artifact_path="model")

    cm = confusion_matrix(y_test, preds_test)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["No fraude", "Fraude"]).plot(ax=ax, cmap="Blues")
    fig.savefig("confusion_matrix_offline.png", bbox_inches="tight")
    mlflow.log_artifact("confusion_matrix_offline.png")
    plt.show()

In [ ]:
joblib.dump({"model": champion, "features": FEATURES}, MODEL_PATH)
print("Modelo guardado en", MODEL_PATH)

production = production.copy()
production["fraud_proba"] = champion.predict_proba(production[FEATURES])[:, 1]
production["fraud_pred"] = (production["fraud_proba"] >= 0.5).astype(int)
production.to_csv(PRODUCTION_DATA_PATH.parent / "production_scored.csv", index=False)
print("Produccion scoreada guardada para el notebook de monitoreo.")

## 2.4 Siguiente paso

- Ver experimentos: `mlflow ui --backend-store-uri ./mlruns` (desde la raiz del repo).
- Continuar con [`3.0-interpretabilidad.ipynb`](3.0-interpretabilidad.ipynb).